In [1]:
import os
from pathlib import Path
# Change cwd to the project root (parent of 'notebooks/')
os.chdir(Path.cwd().parent)
Path.cwd()

PosixPath('/Users/jbrandt/code/birddog')

In [2]:
from birddog.database import (
    Database,
)
from birddog.nocodb_database import (
    clone_table_schema,
    copy_records,
    rename_field,
    copy_formula_field,
    list_formula_fields,
    list_lookup_fields,
    create_lookup_field,
)

2026-05-05 11:55:23,403 [INFO] Using local nocodb api: http://localhost:8080


In [3]:
local_db = Database()
local_db._host

2026-05-05 11:55:23,428 [INFO] 
service throttle report:
  host_key                            cfg_rps  act_rps   tokens  blocked_s max_in_flight
  ----------------------------------------------------------------------------------------
  localhost:api                         20.00    93.68    39.00       0.00           24


'http://localhost:8080'

In [4]:
nocodb_aws_host = os.environ["BIRDDOG_AWS_NOCODB_HOST"]
nocodb_aws_token = os.environ["BIRDDOG_AWS_NOCODB_API_TOKEN"]
nocodb_aws_base_id = os.environ["BIRDDOG_AWS_BASE_ID"]

In [5]:
#nocodb_aws_host = nocodb_aws_host.replace("https", "http")
#nocodb_aws_host

In [6]:
aws_db = Database(host=nocodb_aws_host, api_token=nocodb_aws_token, base_id=nocodb_aws_base_id)
aws_db._host

'http://nocodb-env.eba-xhmfyydr.us-east-2.elasticbeanstalk.com'

In [15]:
def _delete_all(db, table_name):
    db.delete(table_name, db.get_all_ids(table_name))
    
def clone_db(src_db, dest_db):
    try:
        clone_table_schema(src_db, "Schema", dest_db, "Schema")
    except ValueError as err:
        print("Schema table already exists")
    try:    
        clone_table_schema(src_db, "Schema Values", dest_db, "Schema Values")
    except ValueError as err:
        print("Schema Values table already exists")

    _delete_all(dest_db, "Schema")
    copy_records(src_db, "Schema", dest_db, "Schema")

    _delete_all(dest_db, "Schema Values")
    copy_records(src_db, "Schema Values", dest_db, "Schema Values")

    dest_db.load_schema()
    clone_table_schema(src_db, "Pages", dest_db, "Pages")
    clone_table_schema(src_db, "Documents", dest_db, "Documents")
    clone_table_schema(src_db, "MasterFileSummary", dest_db, "MasterFileSummary")
    clone_table_schema(src_db, "FondSummary", dest_db, "FondSummary")
    clone_table_schema(src_db, "OpusSummary", dest_db, "OpusSummary")
    dest_db.load_schema()

In [16]:
clone_db(aws_db, local_db)

Schema table already exists
Schema Values table already exists
2026-05-05 12:00:07,824 [INFO] 
service throttle report:
  host_key                            cfg_rps  act_rps   tokens  blocked_s max_in_flight
  ----------------------------------------------------------------------------------------
  localhost:api                         22.00     0.01    39.00       0.00           24
  nocodb.internal:api                   25.00     0.00    39.00       0.00           24


In [ ]:
rename_field(local_db, local_db._field_id("Pages", "Pages"), "parent")
rename_field(local_db, local_db._field_id("Documents", "Pages"), "owning_pages")
local_db.load_schema()

In [24]:
list_lookup_fields(aws_db, "Documents")

2026-05-05 12:09:57,398 [INFO] 
service throttle report:
  host_key                            cfg_rps  act_rps   tokens  blocked_s max_in_flight
  ----------------------------------------------------------------------------------------
  localhost:api                         27.00     0.03    38.58       0.00           24
  nocodb.internal:api                   27.00     0.02    39.00       0.00           24


['root_label',
 'label',
 'seq_label',
 'page_description',
 'page_native_description',
 'level']

In [13]:
def list_rollup_fields(db, table):
    info = db._get_table_info(table)
    return [c["title"] for c in info["columns"] if c["uidt"] == "Rollup"]

In [14]:
list_rollup_fields(aws_db, "OpusSummary")

2026-05-05 11:58:08,523 [INFO] 
service throttle report:
  host_key                            cfg_rps  act_rps   tokens  blocked_s max_in_flight
  ----------------------------------------------------------------------------------------
  localhost:api                         21.00     0.00    26.94       0.00           24
  nocodb.internal:api                   25.00     0.03    39.00       0.00           24


['priority:files_to_acquire',
 'priority:files_processed',
 'priority:pages_processed']

In [22]:
rename_field(local_db, local_db._field_id("Documents", "Pages"), "owning_pages")

In [25]:
local_db.load_schema()


2026-05-05 12:29:31,235 [INFO] 
service throttle report:
  host_key                            cfg_rps  act_rps   tokens  blocked_s max_in_flight
  ----------------------------------------------------------------------------------------
  localhost:api                         28.00     0.00    39.00       0.00           24
  nocodb.internal:api                   27.00     0.00    39.00       0.00           24


In [27]:
list_formula_fields(aws_db, "Pages")

['root_label']

In [29]:
copy_formula_field(aws_db, "Pages", "root_label", local_db, "Pages", "root_label")

2026-05-05 12:31:32,870 [INFO] 
service throttle report:
  host_key                            cfg_rps  act_rps   tokens  blocked_s max_in_flight
  ----------------------------------------------------------------------------------------
  localhost:api                         29.00     0.02    39.00       0.00           24
  nocodb.internal:api                   29.00     0.02    39.00       0.00           24
2026-05-05 12:31:32,942 [INFO] _fetch response (400): {"msg":"Function REGEX_REPLACE is unavailable for your database"}


FailedIO: HTTP 400: {"msg":"Function REGEX_REPLACE is unavailable for your database"}

In [ ]:
list_formula_fields(db, "Documents OLD")

In [ ]:
r=copy_formula(db, "Documents OLD", "domain", db, "Documents", "domain")

In [ ]:
list_lookup_fields(db,"Documents OLD")

In [ ]:
ci = db._get_table_info("Documents OLD")["columns"]

In [ ]:
for c in ci:
    if c["uidt"] == "Lookup":
        print(c["title"])
        print(c)
        break

In [ ]:
db._get_table_id_map()

In [ ]:
l=list_lookup_fields(db,"Documents OLD")

In [ ]:
#r=create_lookup_field(db, "Documents", "root", "owning_pages", "Pages", "root")

In [ ]:
for f in l[1:]:
    create_lookup_field(db, "Documents", f, "owning_pages", "Pages", f)

In [ ]:
r=create_lookup_field(db, "Documents", "page_description", "owning_pages", "Pages", "description")

In [ ]:
r=create_lookup_field(db, "Documents", "native_page_description", "owning_pages", "Pages", "native_description")

In [ ]:
l

In [ ]:
list_lookup_fields(db,"Documents OLD")

In [ ]:
list_lookup_fields(db,"Documents")

In [ ]:
list_lookup_fields(db,"Pages")

In [ ]:
list_formula_fields(db, "Documents OLD")